# Sentinel-2 Zaman Serileriyle Ürün Sınıflandırması - Faz 1

**Öğrenci:** Esmanur Polattimur  
**Numara:** 241230014

Bu notebookta BreizhCrops ve TUM CropTypes veri setlerinin temel analizi yapılmıştır.

## 1. Gerekli kütüphaneler

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

proje_klasoru = Path.cwd()
veri_klasoru = proje_klasoru / "data" / "phase1"

## 2. BreizhCrops veri setini okuma

In [ ]:
# ============================================================
# 2. BREIZHCROPS VERİLERİNİ OKUMA
# ============================================================

breizh_klasoru = veri_klasoru / "breizhcrops"

sinif_tablosu = pd.read_csv(breizh_klasoru / "classmapping.csv", index_col=0)
frh01 = pd.read_csv(breizh_klasoru / "frh01.csv")
frh02 = pd.read_csv(breizh_klasoru / "frh02.csv")
frh03 = pd.read_csv(breizh_klasoru / "frh03.csv")
frh04 = pd.read_csv(breizh_klasoru / "frh04.csv")

print("FRH01 boyutu:", frh01.shape)
print("FRH02 boyutu:", frh02.shape)
print("FRH03 boyutu:", frh03.shape)
print("FRH04 boyutu:", frh04.shape)

print("\nİlk 5 satır:")
print(frh01.head())

## 3. BreizhCrops veri kontrolü ve sınıf dağılımı

In [ ]:
# ============================================================
# 3. VERİLERİ BİRLEŞTİRME VE KONTROL ETME
# ============================================================

frh01["region"] = "FRH01"
frh02["region"] = "FRH02"
frh03["region"] = "FRH03"
frh04["region"] = "FRH04"

breizh_df = pd.concat([frh01, frh02, frh03, frh04], ignore_index=True)

print("Toplam veri boyutu:", breizh_df.shape)
print("Toplam eksik değer:", breizh_df.isnull().sum().sum())
print("Tekrar eden id:", breizh_df["id"].duplicated().sum())
print("\nBölgelere göre satır sayısı:")
print(breizh_df["region"].value_counts())

In [ ]:
# ============================================================
# 4. SINIF İSİMLERİNİ EKLEME
# ============================================================

kod_sinif = sinif_tablosu.drop_duplicates("code").set_index("code")["classname"]
breizh_df["classname"] = breizh_df["CODE_CULTU"].map(kod_sinif)

# Sınıfı olmayan kayıtları modelde kullanmayacağız.
breizh_eslesen = breizh_df.dropna(subset=["classname"]).copy()

print("Sınıfa eşlenen kayıt:", len(breizh_eslesen))
print("Sınıfa eşlenmeyen kayıt:", breizh_df["classname"].isnull().sum())
print("\nSınıf sayıları:")
print(breizh_eslesen["classname"].value_counts())

In [ ]:
# ============================================================
# 5. BREIZHCROPS SINIF GRAFİĞİ
# ============================================================

breizh_sayim = pd.crosstab(breizh_eslesen["classname"], breizh_eslesen["region"])

breizh_sayim.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("BreizhCrops Sınıf Dağılımı")
plt.xlabel("Ürün sınıfı")
plt.ylabel("Parsel sayısı")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 4. Gerçek bir BreizhCrops parselini inceleme

In [ ]:
# ============================================================
# 6. ÖRNEK PARSEL ZAMAN SERİSİ
# ============================================================

ornek_yol = proje_klasoru / "breizhcrops_frh04_evidence" / "6082638.csv"
ornek_parsel = pd.read_csv(ornek_yol)
ornek_parsel["date"] = pd.to_datetime(ornek_parsel["doa"])
ornek_parsel = ornek_parsel.sort_values("date")

print("Örnek parsel boyutu:", ornek_parsel.shape)
print("Eksik değer:", ornek_parsel.isnull().sum().sum())
print(ornek_parsel.head())

plt.figure(figsize=(12, 5))
for bant in ["B2", "B3", "B4", "B8"]:
    plt.plot(ornek_parsel["date"], ornek_parsel[bant] * 0.0001, label=bant)

plt.title("6082638 Numaralı Parselin Zaman Serisi")
plt.xlabel("Tarih")
plt.ylabel("Yansıtım değeri")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 5. TUM CropTypes veri setini okuma ve analiz etme

In [ ]:
# ============================================================
# 7. TUM CROPTYPES VERİLERİNİ OKUMA
# ============================================================

tum_klasoru = veri_klasoru / "tum_croptypes"
tum_train = pd.read_excel(tum_klasoru / "data2016-2018.xlsx")
tum_test = pd.read_excel(tum_klasoru / "TestData.xlsx")

print("Eğitim veri boyutu:", tum_train.shape)
print("Test veri boyutu:", tum_test.shape)
print("\nEğitim eksik değer:", tum_train.isnull().sum().sum())
print("Test eksik değer:", tum_test.isnull().sum().sum())
print("\nEğitim tekrar eden satır:", tum_train.duplicated().sum())
print("Test tekrar eden satır:", tum_test.duplicated().sum())

In [ ]:
# ============================================================
# 8. TUM SINIFLARINI DÜZENLEME
# ============================================================

kod_degistirme = {
    601: 600, 602: 600, 476: 131, 411: 400,
    171: 400, 410: 400, 177: 400, 489: 311
}

siniflar = {
    600: "patates",
    131: "kışlık arpa",
    400: "mısır",
    311: "kışlık kolza",
    115: "kışlık buğday",
    603: "şeker pancarı",
    1: "diğer"
}

for df in [tum_train, tum_test]:
    df["mapped_code"] = df["NC"].astype(int).replace(kod_degistirme)
    ana_kodlar = [600, 131, 400, 311, 115, 603]
    df.loc[~df["mapped_code"].isin(ana_kodlar), "mapped_code"] = 1
    df["mapped_class"] = df["mapped_code"].map(siniflar)

print("Eğitim sınıfları:")
print(tum_train["mapped_class"].value_counts())
print("\nTest sınıfları:")
print(tum_test["mapped_class"].value_counts())

In [ ]:
# ============================================================
# 9. TUM SINIF GRAFİĞİ
# ============================================================

train_sayim = tum_train["mapped_class"].value_counts()
test_sayim = tum_test["mapped_class"].value_counts()

tum_sayim = pd.DataFrame({
    "Eğitim/geliştirme": train_sayim,
    "Bağımsız test": test_sayim
}).fillna(0)

tum_sayim.plot(kind="bar", stacked=True, figsize=(11, 6))
plt.title("TUM CropTypes Sınıf Dağılımı")
plt.xlabel("Ürün sınıfı")
plt.ylabel("Parsel sayısı")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Kısa sonuç

- BreizhCrops veri setinde sınıf dağılımı dengeli değildir.
- Ayçiçeği ve kabuklu yemiş sınıflarında çok az örnek vardır.
- TUM bağımsız test verisinde şeker pancarı yoktur ve patates yalnız bir örnektir.
- Bu nedenle model sonuçlarında yalnız accuracy değil, Macro-F1 ve sınıf bazlı sonuçlar da kullanılmalıdır.